In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

## Data Wrangling

### Carga de Datos

In [2]:
df = pd.read_csv("data/GBvideos.csv", header=0)

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7993 entries, 0 to 7992
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   video_id        7993 non-null   object 
 1   title           7993 non-null   object 
 2   channel_title   7993 non-null   object 
 3   category_id     7993 non-null   int64  
 4   tags            7993 non-null   object 
 5   views           7993 non-null   int64  
 6   likes           7993 non-null   int64  
 7   dislikes        7993 non-null   int64  
 8   comment_total   7993 non-null   int64  
 9   thumbnail_link  7993 non-null   object 
 10  date            7993 non-null   float64
dtypes: float64(1), int64(5), object(5)
memory usage: 687.0+ KB


In [4]:
df.shape

(7993, 11)

### EDA

In [5]:
df.head()

,video_id,title,channel_title,category_id,tags,views,likes,dislikes,comment_total,thumbnail_link,date
0,jt2OHQh0HoQ,Live Apple Event - Apple September Event 2017 ...,Apple Event,28,apple events|apple event|iphone 8|iphone x|iph...,7426393,78240,13548,705,https://i.ytimg.com/vi/jt2OHQh0HoQ/default_liv...,13.09
1,AqokkXoa7uE,Holly and Phillip Meet Samantha the Sex Robot ...,This Morning,24,this morning|interview|holly willoughby|philli...,494203,2651,1309,0,https://i.ytimg.com/vi/AqokkXoa7uE/default.jpg,13.09
2,YPVcg45W0z4,My DNA Test Results! I'm WHAT?!,emmablackery,24,emmablackery|emma blackery|emma|blackery|briti...,142819,13119,151,1141,https://i.ytimg.com/vi/YPVcg45W0z4/default.jpg,13.09
3,T_PuZBdT2iM,getting into a conversation in a language you ...,ProZD,1,skit|korean|language|conversation|esl|japanese...,1580028,65729,1529,3598,https://i.ytimg.com/vi/T_PuZBdT2iM/default.jpg,13.09
4,NsjsmgmbCfc,Baby Name Challenge!,Sprinkleofglitter,26,sprinkleofglitter|sprinkle of glitter|baby gli...,40592,5019,57,490,https://i.ytimg.com/vi/NsjsmgmbCfc/default.jpg,13.09


In [6]:
df.describe()

,category_id,views,likes,dislikes,comment_total,date
count,7993.000000,7.993000e+03,7.993000e+03,7993.000000,7993.000000,7993.000000
mean,19.738271,1.110733e+06,3.885966e+04,1528.858877,4991.631177,16.088874
std,7.178132,3.048740e+06,1.092951e+05,8176.707788,26837.135518,7.678176
min,1.000000,0.000000e+00,0.000000e+00,0.000000,0.000000,1.100000
25%,17.000000,1.083750e+05,2.580000e+03,71.000000,308.000000,10.100000
50%,23.000000,3.151940e+05,9.561000e+03,249.000000,1038.000000,16.100000
75%,24.000000,9.712210e+05,3.159300e+04,867.000000,3327.000000,21.100000
max,29.000000,5.896141e+07,2.289911e+06,192725.000000,813322.000000,30.090000


In [7]:
df.isnull().sum()

video_id          0
title             0
channel_title     0
category_id       0
tags              0
views             0
likes             0
dislikes          0
comment_total     0
thumbnail_link    0
date              0
dtype: int64

In [8]:
varc = ['category_id', 'views', 'likes', 'dislikes', 'comment_total']
vard = ['channel_title','tags','title','date']
vars_drop = ['video_id','thumbnail_link']

In [9]:
df = df.drop(columns=vars_drop)

In [10]:
#Filtrado de datos por registros con comentarios y likes mayores a 0
df = df[(df['comment_total'] > 0) & (df['likes'] > 0)]
df.shape

(7790, 9)

#### Tratamiento de outliers

In [11]:
# Boxplot dinámico por variable continua
def outliers_view(vars):
    fig = go.Figure()
    for i, col in enumerate(vars):
        fig.add_trace(go.Box(y=df[col].dropna(), name=col, boxpoints='outliers', visible=(i==0)))

    buttons = [
        dict(
            label=col,
            method='update',
            args=[{'visible': [j==i for j in range(len(vars))]}, {'title': f'Boxplot - {col}', 'yaxis': {'title': col}}]
        )
        for i, col in enumerate(varc)
    ]

    fig.update_layout(title=f'Boxplot - {vars[0]}', showlegend=False,
                        updatemenus=[dict(active=0, buttons=buttons, x=1.05, y=1)])
    fig.show()

outliers_view(varc)

In [12]:
#Remoción de outliers
for col in varc:
  Q1 = df[col].quantile(0.05)
  Q3 = df[col].quantile(0.95)
  IQR = Q3 - Q1
  lower_bound = Q1 - 1.5 * IQR
  upper_bound = Q3 + 1.5 * IQR
  df = df[(df[col] >= lower_bound) & (df[col] <= upper_bound)]

outliers_view(varc)

In [13]:
#Histograma de variables
for col in varc:
    fig = px.histogram(df.reset_index(), x=col, title=col)
    fig.show()

In [14]:
#creamos las nuevas variables
df["interaction"] = df["comment_total"] / (df["likes"] + 1)

p33 = df["interaction"].quantile(0.33)
p66 = df["interaction"].quantile(0.66)

def clasificar_interaccion(x):
    if x <= p33:
        return "Baja"
    elif x <= p66:
        return "Aceptable"
    else:
        return "Alta"

df["target"] = df["interaction"].apply(clasificar_interaccion)

In [15]:
df.category_id.value_counts()

category_id
24    1374
10    1142
26    1133
22    1103
17     671
23     488
1      376
25     262
20     249
28     248
27     210
15      78
2       63
19      46
29       5
Name: count, dtype: int64

In [16]:
df[["likes", "comment_total"]].corr()

,likes,comment_total
likes,1.000000,0.798946
comment_total,0.798946,1.000000


## Segmentación de sets

In [17]:
# Variable objetivo discreta
tgt = "target"
vars_pred = [x for x in df.columns if x not in [tgt]]

In [18]:
X = df[vars_pred]
y = df[tgt]

## Modelado